# 复现包路径适配与历史输出

原始来源：`/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision/GRN/bin100/robustness/05_GRN_rep05.ipynb`。
原文件 SHA256：`e771440a9c1d4ced0de5d880602d9f160b82ca4a3120a9d7a0f7394ecf48fbb0`。

**本 notebook 本次未重新执行、未重新训练。以下 execution_count 与全部 cell 输出原样来自原文件，属于历史记录。修改后的路径代码尚未做运行时验证。**

从 `reproducibility` 包根目录运行，`base = Path.cwd()`；配置位于 `manifests/training_manifest.json`。默认读取原外部输入矩阵；所有新训练结果、临时文件、降采样矩阵和 QC 写到包外 `../reproducibility_work`。原始输入、原结果和包内数据不被写回。

推断参数、随机种子、常数基因处理和 AUCell 设置保持原样；两份排名数据库改为清单中的精确文件路径。
路径修改及验证范围见 `manifests/training_notes.md` 和 `manifests/training_notebook_sources.tsv`。


# T577 bin100 counts50：rep05
降采样种子 `20261020`。SpaGRN 原生分步流程：空间共表达 → modules → cisTarget → AUCell。
沿用完整组 Moran、30 邻居、四项 FDR 交集和同一参考数据库。训练排除零方差基因，保存结果保留全部 bin 和基因列。
[官方教程](https://spagrn.readthedocs.io/en/latest/content/01_Basic_Usage.html)。所有绘图集中在 `08_visualization.ipynb`。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import scanpy as sc
from spagrn.regulatory_network import InferNetwork

base = Path.cwd()
manifest = json.loads((base / "manifests/training_manifest.json").read_text())
run = manifest["runs"][4]
out = base / run["output_dir"]
out.mkdir(parents=True, exist_ok=True)
adata = sc.read_h5ad(base / run["input"])
counts = adata.X.astype("float64").tocsc()
variable = np.asarray(counts.power(2).mean(axis=0)).ravel() > np.asarray(counts.mean(axis=0)).ravel() ** 2
analysis = adata[:, variable].copy()
analysis.layers["raw_counts"] = counts[:, variable]
adata.var["spagrn_variable_for_training"] = variable
pd.DataFrame([dict(run=run["run"], seed=run["seed"], bins=adata.n_obs,
                   genes=adata.n_vars, training_genes=analysis.n_vars)])

/data/lvyongji/envs/conda_envs/spagrn/lib/python3.8/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


,run,seed,bins,genes,training_genes
0,rep05,20261020,117867,15541,15537


In [2]:
reference = manifest["reference"]
grn = InferNetwork(analysis, project_name=f"T577_SpaGRN_{run['run']}")
grn.tmp_dir = str(out / "tmp_files")
Path(grn.tmp_dir).mkdir(exist_ok=True)
tfs = grn.load_tfs(str(base / reference["tf_list_file"]))
dbs = [database for ranking_file in reference["ranking_files"]
       for database in grn.load_database(ranking_file)]

In [3]:
adj = grn.spg(
    analysis, tf_list=tfs, layer_key="raw_counts", latent_obsm_key="spatial",
    umi_counts_obs_key="total_counts", model="bernoulli", n_neighbors=30,
    weighted_graph=False, jobs=16, local=False, combine=False,
    methods=["FDR_C", "FDR_I", "FDR_G", "FDR"], operation="intersection",
    mode="moran", torch_device="cuda:0", save_tmp=True, cache=False,
    fn=str(out / f"T577_SpaGRN_{run['run']}.adjacencies.csv"),
)

[spg] Entering spg method
[spg] No cached file found, computing new correlations
[spg] No gene_list provided, computing from hotspot
[spg] Creating KNN graph...


[spg] KNN graph created in 22.03 seconds
[spg] Computing autocorrelations...


  0%|          | 0/15537 [00:00<?, ?it/s]

  0%|          | 1/15537 [00:00<4:12:41,  1.02it/s]

  0%|          | 8/15537 [00:01<26:37,  9.72it/s]  

  0%|          | 15/15537 [00:01<13:46, 18.78it/s]

... (2,226 outputs left out)

[spg] Computing global bivariate Moran...


run_in_parallel_m: Total time taken: 9112.8974 seconds


[spg] Local correlations computed in 2.53 hours (32.08 minutes)
[spg] Processing local_correlations...
[spg] Saving local_correlations to file...


[spg] Exiting spg method


In [4]:
matrix = adata.to_df()
modules = grn.get_modules(adj, matrix, rho_mask_dropouts=False, save_tmp=True, cache=False)
regulons = grn.prune_modules(
    modules, dbs, reference["motif_annotation_file"], num_workers=8,
    rank_threshold=1500, auc_threshold=0.05, nes_threshold=3.0,
    motif_similarity_fdr=0.05, save_tmp=True, cache=False, fn=str(base / run["motifs"]),
)

[get_modules] Entering get_modules method
[get_modules] Computing modules from adjacencies...



2026-10-27 03:05:10,209 - pyscenic.utils - INFO - Calculating Pearson correlations.



2026-10-27 03:05:10,671 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].



2026-10-27 03:06:21,651 - pyscenic.utils - INFO - Creating modules.


[get_modules] Created 2045 modules in 1.42 minutes (85.03 seconds)
[get_modules] Saving modules to file...


[get_modules] Exiting get_modules method


... (1,991 outputs left out)

[#####################################   ] | 92% Completed | 205.40 s

[######################################  ] | 96% Completed | 205.51 s

[########################################] | 100% Completed | 205.62 s

Motif enrichment computed in 3.45 minutes (206.86 seconds)
Create regulons from a dataframe of enriched features.
Additional columns saved: []


In [5]:
auc = grn.cal_auc(matrix, regulons, auc_threshold=0.05, num_workers=1,
                 seed=20261014, normalize=False, cache=False, save_tmp=False)
adata.obsm["auc_mtx"] = auc.reindex(adata.obs_names)
adata.uns["adj"] = adj
adata.uns["regulon_dict"] = grn.regulon_dict
adata.uns["method"] = "SpaGRN"
adata.uns["spagrn_reference_species"] = reference["species"]
adata.uns["spagrn_parameters"] = dict(mode="moran", n_neighbors=30,
    methods=["FDR_C", "FDR_I", "FDR_G", "FDR"], operation="intersection",
    rank_threshold=1500, prune_auc_threshold=0.05, nes_threshold=3.0, auc_threshold=0.05)
adata.write_h5ad(base / run["result"], compression="gzip")
adata.obsm["auc_mtx"].to_csv(out / f"T577_SpaGRN_{run['run']}.aucell_all_bins.csv.gz")
pd.DataFrame([dict(run=run["run"], bins=adata.n_obs, genes=adata.n_vars,
    regulons=auc.shape[1], adjacency_rows=len(adj), total_counts=int(adata.obs["total_counts"].sum()))])

  0%|          | 0/212 [00:00<?, ?it/s]

  0%|          | 1/212 [00:04<14:05,  4.01s/it]

  1%|          | 2/212 [00:04<06:28,  1.85s/it]

  1%|▏         | 3/212 [00:04<04:12,  1.21s/it]

  2%|▏         | 4/212 [00:05<03:06,  1.11it/s]

  2%|▏         | 5/212 [00:06<03:27,  1.00s/it]

... (206 outputs left out)

100%|██████████| 212/212 [02:18<00:00,  1.75it/s]

100%|██████████| 212/212 [02:18<00:00,  1.53it/s]

AUCell calculation completed in 3.24 minutes (194.41 seconds)


,run,bins,genes,regulons,adjacency_rows,total_counts
0,rep05,117867,15541,212,4363100,135429514
